# 01_流式与事件流

**来源:**
- [LangChain Docs — Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)
- [LangChain Docs — Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [LangChain Docs — Frontend Streaming](https://docs.langchain.com/oss/python/deepagents/frontend-streaming)
- [LangGraph 流式指南](https://langchain-ai.github.io/langgraph/how-tos/stream-subgraphs/)

本 Notebook 演示 Deep Agents 的流式输出和事件流 API。Deep Agents 基于 LangGraph 的流式基础设施构建，支持 stream 模式和 stream_events v3 投影（messages/values/tool_calls/subgraphs），以及推理内容和子 Agent 流式。

## 目录

1. [环境准备与导入](#1-环境准备与导入)
2. [Stream 模式](#2-stream-模式)
3. [Stream Events v3 投影](#3-stream-events-v3-投影)
4. [子 Agent 流式](#4-子-agent-流式)
5. [推理内容流式](#5-推理内容流式)
6. [组合多种流式模式](#6-组合多种流式模式)
7. [v2 流式格式](#7-v2-流式格式)
8. [并发消费与交错](#8-并发消费与交错)

**参考链接**
- [Deep Agents Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)
- [Deep Agents Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [LangGraph Streaming](https://langchain-ai.github.io/langgraph/concepts/streaming/)
- [子 Agent 前端流式](https://docs.langchain.com/oss/python/deepagents/frontend-streaming)

## 1. 环境准备与导入

In [ ]:
# pip install deepagents langchain langgraph

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

## 2. Stream 模式

`agent.stream()` 支持多种 stream_mode：

| 模式 | 说明 |
|------|------|
| `"updates"` | 每个节点的输出更新 |
| `"messages"` | 逐 Token 消息流（含工具调用） |
| `"custom"` | 自定义事件（通过 `get_stream_writer` 发送） |
| `"values"` | 每个节点的完整状态快照 |
| `["updates", "messages"]` | 组合模式，同时接收多种事件 |

In [ ]:
# 创建基础 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的助手。请用中文回答。",
)

# stream_mode="updates" — 节点级别的更新
print("=== stream_mode='updates' ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "简单介绍一下 Python"}]},
    stream_mode="updates",
):
    for node_name, data in chunk.items():
        if "messages" in data:
            last_msg = data["messages"][-1]
            if hasattr(last_msg, "content") and last_msg.content:
                print(f"[{node_name}] {last_msg.content[:100]}...")

In [ ]:
# stream_mode="messages" — 逐 Token 流式输出
print("=== stream_mode='messages' ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "用一句诗描述编程"}]},
    stream_mode="messages",
):
    token, metadata = chunk
    if token.content:
        print(token.content, end="", flush=True)
print()

## 3. Stream Events v3 投影

Deep Agents v0.6 引入了 `stream_events` API，这是一种类型化投影 API。它提供独立的迭代器用于子 Agent、消息、工具调用和值，无需在 `stream_mode` 块上进行分支判断。

使用 `version="v3"` 格式。

In [ ]:
# 创建带子 Agent 的 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的助手。",
    subagents=[
        {
            "name": "writer",
            "description": "创意写作",
            "system_prompt": "你是一个诗人。请用中文创作。",
        }
    ],
)

# stream_events API — 使用 v3 格式
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于大海的短诗"}]},
    version="v3",
)

# 投影1: messages — 获取消息流
print("=== Messages 投影 ===")
for message in stream.messages:
    print(f"  [消息] {message.text[:100]}...")

# 投影2: values — 获取状态快照
print("=== Values 投影 ===")
for value in stream.values:
    print(f"  [值] {list(value.keys()) if isinstance(value, dict) else type(value).__name__}")

# 投影3: tool_calls — 获取工具调用
print("=== Tool Calls 投影 ===")
for call in stream.tool_calls:
    print(f"  [工具] {call.tool_name}({call.input})")
    if call.completed:
        print(f"  完成: 错误={call.error}, 输出={str(call.output)[:80] if call.output else None}")

# 投影4: subagents — 获取子 Agent
print("=== Subagents 投影 ===")
for subagent in stream.subagents:
    print(f"  [子Agent] {subagent.name}, 路径: {subagent.path}, 状态: {subagent.status}")

### 3.1 stream_events v3 投影字段

`stream_events` 返回的 `Stream` 对象包含以下属性：

| 投影 | 类型 | 说明 |
|------|------|------|
| `stream.messages` | `Iterable[StreamMessage]` | 所有消息（主 Agent） |
| `stream.values` | `Iterable[dict]` | 每个节点的完整状态快照 |
| `stream.tool_calls` | `Iterable[StreamToolCall]` | 工具调用事件 |
| `stream.subagents` | `Iterable[SubagentStream]` | 子 Agent 的流句柄 |
| `stream.subgraphs` | `Iterable[SubgraphStream]` | 子图执行结构 |

每个 SubagentStream 也包含相同的投影：

| 子 Agent 字段 | 类型 | 说明 |
|--------------|------|------|
| `subagent.name` | str | 子 Agent 名称 |
| `subagent.messages` | Iterable | 子 Agent 发出的消息 |
| `subagent.subagents` | Iterable | 嵌套的子 Agent |
| `subagent.output` | Any | 子 Agent 最终输出 |
| `subagent.path` | str | 命名空间路径 |
| `subagent.status` | str | 生命周期状态 |
| `subagent.tool_calls` | Iterable | 子 Agent 的工具调用 |

## 4. 子 Agent 流式

### 4.1 使用 subgraphs=True 启用子图流

In [ ]:
# 启用子图流式
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    stream_mode="updates",
    subgraphs=True,  # 启用子图流式
    version="v2",
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            # 子 Agent 事件
            print(f"[子Agent: {chunk['ns']}] {list(chunk['data'].keys())}")
        else:
            # 主 Agent 事件
            print(f"[主Agent] {list(chunk['data'].keys())}")

### 4.2 Namespace 路由

当启用子图流式后，每个流式事件包含一个 `namespace`（命名空间），用于标识来源：

| Namespace | 来源 |
|-----------|------|
| `()` (空) | 主 Agent |
| `("tools:abc123")` | 主 Agent 的 task 工具调用的子 Agent |
| `("tools:abc123", "model_request:def456")` | 子 Agent 内部的模型请求节点 |

In [ ]:
# 使用 Namespace 路由事件到正确的 UI 组件
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "updates":
        # 判断是否来自子 Agent
        is_subagent = any(
            segment.startswith("tools:") for segment in chunk["ns"]
        )

        if is_subagent:
            tool_call_id = next(
                s.split(":")[1] for s in chunk["ns"] if s.startswith("tools:")
            )
            print(f"[子Agent {tool_call_id}] {chunk['data']}")
        else:
            print(f"[主Agent] {chunk['data']}")

### 4.3 子 Agent Token 流式

使用 `stream_mode="messages"` 从主 Agent 和子 Agent 流式输出独立的 Token。

In [ ]:
# 流式输出 LLM Token（区分主 Agent 和子 Agent）
current_source = ""

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    stream_mode="messages",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        # 判断来源
        is_subagent = any(s.startswith("tools:") for s in chunk["ns"])

        if is_subagent:
            subagent_ns = next(s for s in chunk["ns"] if s.startswith("tools:"))
            if subagent_ns != current_source:
                print(f"\n\n--- [子Agent: {subagent_ns}] ---")
                current_source = subagent_ns
            if token.content:
                print(token.content, end="", flush=True)
        else:
            if "main" != current_source:
                print("\n\n--- [主Agent] ---")
                current_source = "main"
            if token.content:
                print(token.content, end="", flush=True)

print()

## 5. 推理内容流式

某些模型（如 Anthropic Claude）支持推理/思考内容的流式输出。Deep Agents 通过事件流 API 暴露这些内容。

In [ ]:
# 推理内容流式示例（需要 Anthropic 模型）
# 当使用支持推理的模型时，token.metadata 中包含 reasoning_content

# agent = create_deep_agent(
#     model="anthropic:claude-sonnet-4-6",
#     model_params={
#         "thinking": {"type": "enabled", "budget_tokens": 10000}
#     },
# )

# for chunk in agent.stream(
#     {"messages": [{"role": "user", "content": "解决一个复杂数学问题"}]},
#     stream_mode="messages",
# ):
#     token, metadata = chunk
#     if metadata.get("reasoning_content"):
#         print(f"[推理] {metadata['reasoning_content']}", end="", flush=True)
#     elif token.content:
#         print(f"[回答] {token.content}", end="", flush=True)

print("推理内容流式示例（需要 Anthropic Claude + thinking 参数）")

## 6. 组合多种流式模式

可以组合多种流式模式以获得 Agent 执行的完整视图。

In [ ]:
# 组合 updates + messages + custom 三种模式
INTERESTING_NODES = {"model_request", "tools"}

last_source = ""
mid_line = False

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    stream_mode=["updates", "messages", "custom"],
    subgraphs=True,
    version="v2",
):
    is_subagent = any(s.startswith("tools:") for s in chunk["ns"])
    source = "子Agent" if is_subagent else "主"

    if chunk["type"] == "updates":
        for node_name in chunk["data"]:
            if node_name not in INTERESTING_NODES:
                continue
            if mid_line:
                print()
                mid_line = False
            print(f"[{source}] 步骤: {node_name}")

    elif chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if token.content:
            if source != last_source:
                if mid_line:
                    print()
                    mid_line = False
                print(f"\n[{source}] ", end="")
                last_source = source
            print(token.content, end="", flush=True)
            mid_line = True

    elif chunk["type"] == "custom":
        if mid_line:
            print()
            mid_line = False
        print(f"[{source}] 自定义事件: {chunk['data']}")

print()

## 7. v2 流式格式

所有示例可以使用 `version="v2"` 格式（需要 LangGraph >= 1.1）。

每个块是包含 `type`、`ns` 和 `data` 键的 `StreamPart` 字典——无论流式模式、模式数量或子图设置如何，形状保持一致。

| 字段 | 说明 |
|------|------|
| `type` | 事件类型: `"updates"` / `"messages"` / `"custom"` / `"values"` |
| `ns` | Namespace 路径，标识事件来源 |
| `data` | 事件负载，取决于 type |

In [ ]:
# v2 格式的结构
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Hi"}]},
    stream_mode="messages",
    subgraphs=True,
    version="v2",
):
    print(f"type={chunk['type']}, ns={chunk['ns']}, data_type={type(chunk['data']).__name__}")
    break  # 只显示第一个块

## 8. 并发消费与交错

协调 Agent 和子 Agent 的输出通常是交错的。需要实时 UI 更新时应并发消费投影。

### stream.interleave 交错消费

In [ ]:
# 使用 interleave 交错消费多个投影
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    version="v3",
)

for name, item in stream.interleave("messages", "subagents"):
    if name == "messages":
        print(f"[协调] {item.text[:80]}")
    else:
        for message in item.messages:
            print(f"[{item.name}] {message.text[:80]}")
        break  # 仅演示第一个子 Agent

### 子 Agent vs 子图

| 概念 | 说明 | 使用场景 |
|------|------|---------|
| `stream.subgraphs` | 展示图执行结构（底层） | 调试、监控 |
| `stream.subagents` | 展示产品级的任务委托（高层） | 面向用户的 UI |

对于面向用户的 UI，应使用 `stream.subagents`，因为它隐藏了内部图节点，直接暴露子 Agent 概念。

---

**参考链接**
- [Deep Agents Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)
- [Deep Agents Event Streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [子 Agent 前端流式](https://docs.langchain.com/oss/python/deepagents/frontend-streaming)
- [LangGraph Streaming](https://langchain-ai.github.io/langgraph/concepts/streaming/)
- [LangGraph Stream Subgraphs](https://langchain-ai.github.io/langgraph/how-tos/stream-subgraphs/)